In [0]:
%python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", DoubleType(), False),
])

data = [
    (1, "Alice", "Engineering", 85000.0),
    (2, "Bob", "Sales", 55000.0),
    (3, "Charlie", "Engineering", 92000.0),
    (4, "Diana", "HR", 61000.0),
    (5, "Evan", "Finance", 72000.0),
    (6, "Fiona", "Marketing", 58000.0),
]

employee_df = spark.createDataFrame(data, schema)
employee_df.show()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Find highest paid employee
windowSpec = Window.partitionBy("department").orderBy(F.col("salary").desc())
employee_df.withColumn("rn",F.row_number().over(windowSpec)).filter(F.col("rn")==1) \
           .withColumn("drnk",F.dense_rank().over(windowSpec)).filter(F.col("drnk")==1) \
           .withColumn("prev_salary",F.lag("salary",1).over(windowSpec)) \
           .withColumn("next_salary",F.lag("salary",-1).over(windowSpec)).drop("rn","drnk").show()

In [0]:
cumulative_window = Window.partitionBy("department").orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
employee_df.withColumn("cumulative_salary", F.sum("salary").over(cumulative_window)).show()